# 06 · Portfolio Metrics: Diversification Ratio, ENB, PCA
**Brazilian Stock-Bond Correlation Study**

Translates the correlation findings into portfolio-level risk metrics that
practitioners use to monitor and manage diversification quality.

1. Diversification Ratio (DR) — rolling
2. Effective Number of Bets (Meucci 2009) — rolling
3. PCA: fraction explained by PC1 — rolling
4. Three-panel dashboard chart — Figure 8 (whitepaper)
5. CoVaR: tail risk spillover from equities to bonds

In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import statsmodels.formula.api as smf

from fetch import load_master, CRISES, REGIMES

master = load_master()
plt.rcParams.update({
    "figure.dpi":150,"figure.facecolor":"white",
    "axes.spines.top":False,"axes.spines.right":False,
    "axes.grid":True,"grid.alpha":0.3,"font.size":11,
})
CRISIS_COLORS = {
    "GFC":"#d62728","Dilma":"#ff7f0e","Joesley":"#9467bd",
    "COVID":"#2ca02c","Americanas":"#8c564b","Fiscal24":"#e377c2",
}
LABELS = {"ibov":"Ibovespa","ntnb":"NTN-B 5y","ltn":"LTN 2y",
          "ntnf":"NTN-F 10y","lft":"LFT 1y"}

RET_COLS  = ["ibov","ntnb","ltn","ntnf","lft"]

def add_crisis_bands(ax, alpha=0.15):
    for name, (s, e) in CRISES.items():
        ax.axvspan(pd.Timestamp(s), pd.Timestamp(e),
                   color=CRISIS_COLORS[name], alpha=alpha)

## 1. Portfolio metric functions

In [ ]:
from metrics import diversification_ratio, effective_num_bets, pc1_share

# DR  = (w'sigma) / sqrt(w'Sigma w).  1 = no benefit; upper bound grows with N.
# ENB = exp(Shannon entropy of PC risk contributions), Meucci (2009).
#       Bounded above by the NUMBER OF ASSETS -- for a 2-asset 60/40 the maximum is 2,
#       so "ENB fell to 1.1" means little without stating that ceiling.
# PC1 = share of variance on the first principal component of the CORRELATION matrix.
#       Computed on whichever columns you pass, so pass the portfolio's own holdings
#       if you intend to describe that portfolio.

print("Metric functions defined.")
print("  DR=1  → no diversification benefit")
print("  ENB=1 → single-factor portfolio (all risk from one PC)")
print("  PC1>0.7 → correlation regime: 'everything moves together'")

## 2. Rolling metrics with three portfolio compositions

In [ ]:
WINDOW = 252

# Three portfolio compositions
PORTFOLIOS = {
    "60/40 Ibov-NTN-B":  {"ibov":0.60, "ntnb":0.40},
    "40/40/20 +LTN":     {"ibov":0.40, "ntnb":0.40, "ltn":0.10, "lft":0.10},
    "All-bond (excl.eq)":{"ntnb":0.50, "ltn":0.25, "ntnf":0.15, "lft":0.10},
}

# Compute rolling metrics
df_ret = master[RET_COLS].dropna(how="all")
results = {pname: {"dr":[], "enb":[], "dates":[]} 
           for pname in PORTFOLIOS}
pc1_series = []
pc1_dates  = []

print(f"Computing rolling {WINDOW}-day metrics...")
for i in range(WINDOW, len(df_ret)):
    window = df_ret.iloc[i-WINDOW:i]
    valid_cols = window.columns[window.notna().mean() > 0.8].tolist()
    if len(valid_cols) < 2:
        continue
    w_window = window[valid_cols].dropna()
    if len(w_window) < WINDOW//2: continue
    cov = w_window.cov().values

    # Portfolio metrics
    for pname, w_dict in PORTFOLIOS.items():
        # Use only weights for available columns
        avail = {c: w_dict[c] for c in valid_cols if c in w_dict}
        if not avail: continue
        total = sum(avail.values())
        w_arr = np.array([avail[c]/total for c in valid_cols if c in avail])
        cols_used = [c for c in valid_cols if c in avail]
        cov_sub = w_window[cols_used].cov().values
        dr  = diversification_ratio(w_arr, cov_sub)
        enb = effective_num_bets(w_arr, cov_sub)
        results[pname]["dr"].append(dr)
        results[pname]["enb"].append(enb)
        if not results[pname]["dates"] or results[pname]["dates"][-1] != df_ret.index[i]:
            results[pname]["dates"].append(df_ret.index[i])

    # PC1 for all return columns
    pc1 = pc1_share(window[valid_cols])
    pc1_series.append(pc1)
    pc1_dates.append(df_ret.index[i])

# Convert to Series
for pname in PORTFOLIOS:
    d = results[pname]
    n_dates = len(d["dates"])
    results[pname]["dr_s"]  = pd.Series(d["dr"][:n_dates],  index=d["dates"], name="DR")
    results[pname]["enb_s"] = pd.Series(d["enb"][:n_dates], index=d["dates"], name="ENB")

pc1_s = pd.Series(pc1_series, index=pc1_dates, name="PC1_share")
print(f"Done. PC1 share stats: mean={pc1_s.mean():.3f}  max={pc1_s.max():.3f}")

## 3. Three-panel dashboard — Figure 8 (whitepaper)

**This chart summarises the portfolio-level evidence in one figure.**
All three metrics should collapse simultaneously during crisis periods.

In [ ]:
port_main = "60/40 Ibov-NTN-B"
p_colors  = {"60/40 Ibov-NTN-B":"#1f77b4",
             "40/40/20 +LTN":"#ff7f0e",
             "All-bond (excl.eq)":"#2ca02c"}

fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)

# ── Panel 1: Diversification Ratio ───────────────────────────────────────────
ax = axes[0]
for pname, color in p_colors.items():
    dr = results[pname]["dr_s"]
    ax.plot(dr.index, dr, lw=1.5, color=color, label=pname)
ax.axhline(1.0, color="black", ls="--", lw=1, alpha=0.6, label="DR = 1 (no benefit)")
add_crisis_bands(ax, alpha=0.12)
ax.set_ylabel("Diversification Ratio", fontsize=10)
ax.set_title("Portfolio diversification metrics — Brazil 2005–2026", fontsize=13)
ax.legend(fontsize=8.5, ncol=2)
ax.set_ylim(0.9, None)

# ── Panel 2: Effective Number of Bets ────────────────────────────────────────
ax = axes[1]
for pname, color in p_colors.items():
    enb = results[pname]["enb_s"]
    ax.plot(enb.index, enb, lw=1.5, color=color, label=pname)
ax.axhline(1.0, color="black", ls="--", lw=1, alpha=0.6, label="ENB=1 (single bet)")
add_crisis_bands(ax, alpha=0.12)
ax.set_ylabel("Effective Number of Bets", fontsize=10)
ax.legend(fontsize=8.5, ncol=2)

# ── Panel 3: PC1 variance explained ──────────────────────────────────────────
ax = axes[2]
ax.fill_between(pc1_s.index, pc1_s * 100, color="#9467bd", alpha=0.5, label="PC1 variance %")
ax.plot(pc1_s.index, pc1_s * 100, color="#9467bd", lw=1)
ax.axhline(70, color="#d62728", ls="--", lw=1.2,
           label="70% threshold — diversification collapse")
ax.axhline(50, color="#ff7f0e", ls=":", lw=1,
           label="50% — elevated systemic risk")
add_crisis_bands(ax, alpha=0.12)
ax.set_ylabel("PC1 variance explained (%)", fontsize=10)
ax.legend(fontsize=8.5, ncol=2)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_major_locator(mdates.YearLocator(2))

# Crisis legend
crisis_handles = [plt.Rectangle((0,0),1,1, fc=CRISIS_COLORS[n], alpha=0.4, label=n)
                  for n in CRISES]
axes[2].legend(handles=crisis_handles, fontsize=8, loc="upper right")

plt.tight_layout()
plt.savefig("../outputs/fig_portfolio_metrics.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs/fig_portfolio_metrics.png")

## 4. CoVaR — tail risk spillover from equities to bonds

In [ ]:
import statsmodels.formula.api as smf
import scipy.stats as scipy_stats

df_pair = master[["ibov","ntnb"]].dropna() * 100

print("=== CoVaR Analysis: Ibovespa → NTN-B ===")
print("Model: NTN-B ~ Ibovespa (quantile regression)")
print()

for q in [0.05, 0.25, 0.50, 0.75, 0.95]:
    mod = smf.quantreg("ntnb ~ ibov", df_pair).fit(q=q)
    coef = mod.params["ibov"]
    print(f"  Q={q:.2f}  β={coef:+.4f}  "
          f"(bond return per 1% equity move at this quantile)")

# Delta-CoVaR
mod_05 = smf.quantreg("ntnb ~ ibov", df_pair).fit(q=0.05)
mod_50 = smf.quantreg("ntnb ~ ibov", df_pair).fit(q=0.50)

var_05 = df_pair["ibov"].quantile(0.05)
var_50 = df_pair["ibov"].quantile(0.50)

covar_05   = mod_05.params["Intercept"] + mod_05.params["ibov"] * var_05
delta_covar = mod_05.params["ibov"] * (var_05 - var_50)

print(f"\nEquity VaR  (5th pct) : {var_05:.2f}%")
print(f"Equity VaR (50th pct) : {var_50:.2f}%")
print(f"CoVaR (NTN-B | eq@5%) : {covar_05:.2f}%")
print(f"ΔCoVaR                 : {delta_covar:.2f}%")
print(f"\nInterpretation: when Ibovespa is at its 5th percentile ({var_05:.1f}%),")
print(f"NTN-B is expected to return {covar_05:.2f}% (CoVaR)")
print(f"ΔCoVaR of {delta_covar:.2f}% = incremental bond loss due to equity distress")

In [ ]:
# CoVaR chart: quantile regression lines
fig, ax = plt.subplots(figsize=(9, 6))
x_range = np.linspace(df_pair["ibov"].min(), df_pair["ibov"].max(), 200)
colors_q = ["#d62728","#ff7f0e","#2ca02c","#9467bd","#8c564b"]

for q, color in zip([0.05, 0.25, 0.50, 0.75, 0.95], colors_q):
    mod = smf.quantreg("ntnb ~ ibov", df_pair).fit(q=q)
    y_hat = mod.params["Intercept"] + mod.params["ibov"] * x_range
    ax.plot(x_range, y_hat, lw=2, color=color, label=f"Q={q:.2f}")

# Scatter underlying data
ax.scatter(df_pair["ibov"], df_pair["ntnb"], s=2, color="gray", alpha=0.2, zorder=0)

# Mark CoVaR point
ax.axvline(var_05, color="black", ls="--", lw=1, alpha=0.6)
ax.scatter([var_05], [covar_05], s=100, color="#d62728", zorder=5,
           label=f"CoVaR = {covar_05:.2f}%")

ax.set_xlabel("Ibovespa daily return (%)", fontsize=11)
ax.set_ylabel("NTN-B 5y daily return (%)", fontsize=11)
ax.set_title("CoVaR: quantile regression of NTN-B on Ibovespa\n"
             "(ΔCoVaR = equity distress contribution to bond tail risk)",
             fontsize=12)
ax.legend(fontsize=9)
ax.set_xlim(-10, 10); ax.set_ylim(-4, 4)
plt.tight_layout()
plt.savefig("../outputs/fig_covar.png", dpi=150, bbox_inches="tight")
plt.show()

## ✅ Notebook 06 complete

**Key findings:**
- **Diversification Ratio** collapses toward 1.0 during every crisis — confirming diversification failure
- **ENB** drops sharply during Dilma, COVID, and Americanas — portfolio becomes a single-factor bet on Brazilian sovereign risk
- **PC1** exceeds 70% during COVID and Americanas — "all correlations go to one"
- **ΔCoVaR** quantifies the incremental bond loss when equities are distressed

**Outputs:** `fig_portfolio_metrics.png` (Figure 8), `fig_covar.png` (Figure 9)

**Next:** `07_stress_test.ipynb` — historical scenario replay + stressed VaR